In [1]:
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, chi2, f_classif
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../../')

from src.utils import helpers as func
import prince
import plotly.express as px
! pip install holidays

In [2]:
path = '../../data/processed/df_entidades_acum.parquet'
df = func.load_data(path)

Cargando datos desde: ../../data/processed/df_entidades_acum.parquet
Dimensiones del DataFrame: (18272, 4)

Primeras filas del DataFrame:


,Estado,Fecha,Valor,Valor_acumulado
0,Aguascalientes,2014-01-06,1,1
1,Aguascalientes,2014-01-13,2,3
2,Aguascalientes,2014-01-20,1,4
3,Aguascalientes,2014-01-27,1,5
4,Aguascalientes,2014-02-03,2,7


In [3]:
df = func.clean_column_names(df)
df["año"] = df["fecha"].dt.isocalendar().year
df["semana"] = df["fecha"].dt.isocalendar().week
df['mes'] = df['fecha'].dt.month
df.head()

,estado,fecha,valor,valor_acumulado,año,semana,mes
0,Aguascalientes,2014-01-06,1,1,2014,2,1
1,Aguascalientes,2014-01-13,2,3,2014,3,1
2,Aguascalientes,2014-01-20,1,4,2014,4,1
3,Aguascalientes,2014-01-27,1,5,2014,5,1
4,Aguascalientes,2014-02-03,2,7,2014,6,2


In [4]:
df['trimestre'] = df['fecha'].dt.quarter
df['semestre'] = df['fecha'].dt.month.apply(lambda m: 1 if m <= 6 else 2)
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)
df

,estado,fecha,valor,valor_acumulado,año,semana,mes,trimestre,semestre,mes_sin,mes_cos
0,Aguascalientes,2014-01-06,1,1,2014,2,1,1,1,5.000000e-01,0.866025
1,Aguascalientes,2014-01-13,2,3,2014,3,1,1,1,5.000000e-01,0.866025
2,Aguascalientes,2014-01-20,1,4,2014,4,1,1,1,5.000000e-01,0.866025
3,Aguascalientes,2014-01-27,1,5,2014,5,1,1,1,5.000000e-01,0.866025
4,Aguascalientes,2014-02-03,2,7,2014,6,2,1,1,8.660254e-01,0.500000
...,...,...,...,...,...,...,...,...,...,...,...
18267,Zacatecas,2024-11-25,56,18103,2024,48,11,4,2,-5.000000e-01,0.866025
18268,Zacatecas,2024-12-02,56,18159,2024,49,12,4,2,-2.449294e-16,1.000000
18269,Zacatecas,2024-12-09,47,18206,2024,50,12,4,2,-2.449294e-16,1.000000
18270,Zacatecas,2024-12-16,42,18248,2024,51,12,4,2,-2.449294e-16,1.000000


In [5]:
sem_año = 52
lag_años = [1,2]
for año in lag_años:
    # lag año con respecto al anterior año indicado
    df[f"lag_{año}y"] = df.groupby(['fecha','estado'])["año"].shift(sem_año * año).fillna(0).astype(int)
    # lag del valor con respecto al año anterior indicado
    df[f"lag_v{año}y"] = df.groupby(['fecha','estado'])["valor"].shift(sem_año * año).fillna(0).astype(int)
    # lag del valor acumulado con respecto al año anterior indicado
    df[f"lag_va{año}y"] = df.groupby(['fecha','estado'])["valor_acumulado"].shift(sem_año * año).fillna(0).astype(int)
df

,estado,fecha,valor,valor_acumulado,año,semana,mes,trimestre,semestre,mes_sin,mes_cos,lag_1y,lag_v1y,lag_va1y,lag_2y,lag_v2y,lag_va2y
0,Aguascalientes,2014-01-06,1,1,2014,2,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0
1,Aguascalientes,2014-01-13,2,3,2014,3,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0
2,Aguascalientes,2014-01-20,1,4,2014,4,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0
3,Aguascalientes,2014-01-27,1,5,2014,5,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0
4,Aguascalientes,2014-02-03,2,7,2014,6,2,1,1,8.660254e-01,0.500000,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18267,Zacatecas,2024-11-25,56,18103,2024,48,11,4,2,-5.000000e-01,0.866025,0,0,0,0,0,0
18268,Zacatecas,2024-12-02,56,18159,2024,49,12,4,2,-2.449294e-16,1.000000,0,0,0,0,0,0
18269,Zacatecas,2024-12-09,47,18206,2024,50,12,4,2,-2.449294e-16,1.000000,0,0,0,0,0,0
18270,Zacatecas,2024-12-16,42,18248,2024,51,12,4,2,-2.449294e-16,1.000000,0,0,0,0,0,0


In [6]:
covid_start = pd.to_datetime('2020-03-01')
# covid_end = pd.to_datetime('2022-03-31')
covid_end = pd.to_datetime('2023-05-05')

df['periodo_covid'] = pd.cut(
    df['fecha'],
    bins=[pd.Timestamp.min, covid_start, 
          covid_end, pd.Timestamp.max],
    labels=['Pre-Covid', 'Covid', 'Post-Covid'],
    include_lowest=True
)

df['temporada'] = df['mes'].apply(func.asignar_temporada)
df['festivo'] = df.apply(func.encontrar_festivos_en_semana, axis=1)
df

,estado,fecha,valor,valor_acumulado,año,semana,mes,trimestre,semestre,mes_sin,mes_cos,lag_1y,lag_v1y,lag_va1y,lag_2y,lag_v2y,lag_va2y,periodo_covid,temporada,festivo
0,Aguascalientes,2014-01-06,1,1,2014,2,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0,Pre-Covid,Invierno,0
1,Aguascalientes,2014-01-13,2,3,2014,3,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0,Pre-Covid,Invierno,0
2,Aguascalientes,2014-01-20,1,4,2014,4,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0,Pre-Covid,Invierno,0
3,Aguascalientes,2014-01-27,1,5,2014,5,1,1,1,5.000000e-01,0.866025,0,0,0,0,0,0,Pre-Covid,Invierno,0
4,Aguascalientes,2014-02-03,2,7,2014,6,2,1,1,8.660254e-01,0.500000,0,0,0,0,0,0,Pre-Covid,Invierno,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18267,Zacatecas,2024-11-25,56,18103,2024,48,11,4,2,-5.000000e-01,0.866025,0,0,0,0,0,0,Post-Covid,Otoño,0
18268,Zacatecas,2024-12-02,56,18159,2024,49,12,4,2,-2.449294e-16,1.000000,0,0,0,0,0,0,Post-Covid,Invierno,0
18269,Zacatecas,2024-12-09,47,18206,2024,50,12,4,2,-2.449294e-16,1.000000,0,0,0,0,0,0,Post-Covid,Invierno,0
18270,Zacatecas,2024-12-16,42,18248,2024,51,12,4,2,-2.449294e-16,1.000000,0,0,0,0,0,0,Post-Covid,Invierno,0


In [7]:
df[['semana', 'mes', 'trimestre', 'semestre']] = df[['semana', 'mes', 'trimestre', 'semestre']].astype(object)

In [8]:
columnas_ohe = ['estado','semana','mes','trimestre','semestre','periodo_covid','temporada']
dummies = pd.get_dummies(df[columnas_ohe], drop_first=True)
# Remove columns that overlap before joining
# dummies = dummies.drop(columns=['semana', 'mes', 'trimestre', 'semestre'])
df = df.join(dummies)
df.drop(columns=columnas_ohe, inplace=True)

In [9]:
df.dtypes

fecha                       datetime64[ns]
valor                                int64
valor_acumulado                      int64
año                                 UInt32
mes_sin                            float64
                                 ...      
periodo_covid_Covid                   bool
periodo_covid_Post-Covid              bool
temporada_Otoño                       bool
temporada_Primavera                   bool
temporada_Verano                      bool
Length: 117, dtype: object

In [10]:
df.columns = df.columns.str.strip()               # Elimina espacios al inicio o final
df.columns = df.columns.str.replace(' ', '_')     # Sustituye espacios por guiones bajos
df.columns = df.columns.str.lower()              # Convierte todo a minúsculas para consistencia
df.columns = df.columns.str.replace('_covid_', '_', n=1) # Simplifica nombres de columnas

In [11]:
df

,fecha,valor,valor_acumulado,año,mes_sin,mes_cos,lag_1y,lag_v1y,lag_va1y,lag_2y,...,mes_12,trimestre_2,trimestre_3,trimestre_4,semestre_2,periodo_covid,periodo_post-covid,temporada_otoño,temporada_primavera,temporada_verano
0,2014-01-06,1,1,2014,5.000000e-01,0.866025,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,2014-01-13,2,3,2014,5.000000e-01,0.866025,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,2014-01-20,1,4,2014,5.000000e-01,0.866025,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,2014-01-27,1,5,2014,5.000000e-01,0.866025,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,2014-02-03,2,7,2014,8.660254e-01,0.500000,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18267,2024-11-25,56,18103,2024,-5.000000e-01,0.866025,0,0,0,0,...,False,False,False,True,True,False,True,True,False,False
18268,2024-12-02,56,18159,2024,-2.449294e-16,1.000000,0,0,0,0,...,True,False,False,True,True,False,True,False,False,False
18269,2024-12-09,47,18206,2024,-2.449294e-16,1.000000,0,0,0,0,...,True,False,False,True,True,False,True,False,False,False
18270,2024-12-16,42,18248,2024,-2.449294e-16,1.000000,0,0,0,0,...,True,False,False,True,True,False,True,False,False,False
